# Held-out network catalog-audit checkpoint

This advisor-facing notebook reads **compact tracked products only**. It does not open normalized catalog caches, miniSEED, full score arrays, DAS HDF5, or repeating-family labels.

**Frozen outcome:** all 33 network rows were retained. Six rows are explained by five cataloged earthquakes (one earthquake produced two generic triggers), leaving 27 catalog-unassociated candidates and 32 event-level evaluation units. Catalog absence is not evidence that a candidate is a new earthquake or repeater.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_project(start):
    for candidate in (start, *start.parents):
        if (candidate / 'config' / 'heldout_catalog_audit.json').is_file():
            return candidate
    raise FileNotFoundError('cannot locate repeaters_v2 project root')

PROJECT = find_project(Path.cwd().resolve())
AUDIT = PROJECT / 'outputs' / 'heldout_v2' / 'network_catalog_audit'
NETWORK = PROJECT / 'outputs' / 'heldout_v2' / 'network'
REGISTRATION = PROJECT / 'outputs' / 'heldout_v2' / 'registration'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)

print('Project:', PROJECT)

## Freeze and access-ledger verification

These assertions fail if a frozen input/result changes, a raw network row disappears or changes, or the recorded DAS/family access ledger is nonzero.

In [ ]:
with (AUDIT / 'network_catalog_audit_status.json').open(encoding='utf-8') as handle:
    status = json.load(handle)
with (REGISTRATION / 'catalog_audit_registration_status.json').open(encoding='utf-8') as handle:
    registration = json.load(handle)
with (REGISTRATION / 'catalog_audit_runner_release.json').open(encoding='utf-8') as handle:
    release = json.load(handle)

raw = read_csv(NETWORK / 'network_union_time_only.csv')
adjudicated = read_csv(AUDIT / 'network_union_adjudicated.csv')
units = read_csv(AUDIT / 'network_evaluation_units.csv')
target = read_csv(AUDIT / 'target_catalog_associations.csv')
broader = read_csv(AUDIT / 'broader_catalog_associations.csv')
sources = read_csv(AUDIT / 'broader_catalog_source_ledger.csv')

assert status['status'] == 'PASS'
assert status['stage'] == 'heldout_network_catalog_audit_complete_and_frozen'
assert len(raw) == len(adjudicated) == 33
assert len(units) == 32
for field in raw.columns:
    assert adjudicated[field].tolist() == raw[field].tolist(), field
assert sha256(NETWORK / 'network_union_time_only.csv') == status['network_union_input_sha256']
for stem in ('target_associations', 'broader_source_ledger', 'broader_associations',
             'adjudicated_union', 'evaluation_units'):
    assert sha256(status[stem + '_path']) == status[stem + '_sha256']
assert status['candidate_rows_deleted'] == 0
assert status['candidate_times_edited'] == 0
assert status['family_assignments_made'] == 0
assert status['heldout_DAS_HDF5_files_opened'] == 0
assert status['heldout_DAS_HDF5_datasets_opened'] == 0
assert status['heldout_family_label_rows_opened'] == 0
assert status['heldout_network_waveform_files_opened'] == 0
assert status['broader_catalog_queries_PASS'] == 12
assert release['remote_branch_sha'] == release['runner_commit_sha']

summary = pd.Series({
    'raw network candidates retained': len(adjudicated),
    'event-level evaluation units': len(units),
    'cataloged-event candidate rows': int(adjudicated['known_event_class'].str.startswith('known_').sum()),
    'unique cataloged events': int(adjudicated.loc[
        adjudicated['known_event_class'].str.startswith('known_'),
        'catalog_event_id_final'
    ].nunique()),
    'catalog-unassociated candidates': int(adjudicated['known_event_class'].str.startswith('catalog_unassociated').sum()),
    'catalog conflicts': int(status['catalog_conflict_STOP_count']),
    'target catalog rows audited': int(status['target_catalog_event_rows_opened']),
    'regional catalog rows audited': int(status['broader_catalog_event_rows_opened']),
    'DAS files opened': int(status['heldout_DAS_HDF5_files_opened']),
    'family labels opened': int(status['heldout_family_label_rows_opened']),
})
display(summary.to_frame('frozen value'))

## What the catalog explains

A physical regional association is a veto/audit result, not repeating-family truth. A plausible_match_count above 1 records ambiguity instead of hiding it.

In [ ]:
class_order = [
    'known_family_neighborhood_catalog_event',
    'known_target_box_event_outside_family_neighborhood',
    'known_regional_arrival_outside_target_box',
    'catalog_unassociated_template_only_inconclusive',
    'catalog_unassociated_generic_candidate',
    'catalog_timing_or_identity_conflict_STOP',
]
counts = adjudicated['known_event_class'].value_counts().reindex(class_order, fill_value=0)
display(counts.rename('candidate rows').to_frame())

colors = ['#2471A3', '#17A589', '#7D3C98', '#F5B041', '#E67E22', '#C0392B']
ax = counts.plot.bar(figsize=(10, 4), color=colors)
ax.set_ylabel('frozen candidate-row count')
ax.set_xlabel('')
ax.set_title('Post-union catalog annotation (all 33 rows retained)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
known = adjudicated.loc[
    adjudicated['known_event_class'].str.startswith('known_'),
    [
        'interval_id', 'union_candidate_id', 'generic_trigger_time',
        'broader_catalog_event_id', 'broader_catalog_origin_time',
        'broader_catalog_location_name', 'broader_catalog_magnitude',
        'broader_catalog_path_distance_km', 'broader_catalog_observed_delay_s',
        'broader_catalog_nominal_timing_residual_s',
        'broader_catalog_plausible_match_count', 'known_event_class',
        'evaluation_unit_id'
    ],
].copy()
known['apparent_velocity_km_s'] = (
    pd.to_numeric(known['broader_catalog_path_distance_km'])
    / pd.to_numeric(known['broader_catalog_observed_delay_s'])
)
display(known.round({'apparent_velocity_km_s': 3}))

duplicates = units.loc[pd.to_numeric(units['candidate_count']) > 1]
print('Evaluation units represented by more than one network trigger:')
display(duplicates)

## The unresolved population

The 12 template-only catalog misses are explicitly **inconclusive** because their timestamps are estimated origins, not array arrivals, so the broad physical-arrival veto is not applicable. The 15 generic misses have no plausible event in the registered regional queries. Neither group is automatically a new earthquake.

In [ ]:
interval_classes = pd.crosstab(
    adjudicated['interval_id'],
    adjudicated['known_event_class'],
).reindex(registration['interval_ids'], fill_value=0)
display(interval_classes)

branch_classes = pd.crosstab(
    adjudicated['branch_membership'],
    adjudicated['known_event_class'],
)
display(branch_classes)

ax = interval_classes.plot.bar(stacked=True, figsize=(11, 4), colormap='tab20')
ax.set_ylabel('candidate rows')
ax.set_xlabel('held-out interval')
ax.set_title('Catalog outcomes by independently selected interval')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

## Display-only advisor controls

Edit these values to inspect rows. They do not alter tolerances, candidate identities, catalog associations, or any tracked output.

In [ ]:
DISPLAY_INTERVAL = 'all'  # e.g. 'heldout_08'
DISPLAY_CLASS = 'all'     # e.g. 'catalog_unassociated_generic_candidate'

view = adjudicated.copy()
if DISPLAY_INTERVAL != 'all':
    view = view.loc[view['interval_id'] == DISPLAY_INTERVAL]
if DISPLAY_CLASS != 'all':
    view = view.loc[view['known_event_class'] == DISPLAY_CLASS]

columns = [
    'interval_id', 'union_candidate_id', 'representative_time',
    'branch_membership', 'known_event_class', 'catalog_event_id_final',
    'broader_catalog_location_name', 'broader_catalog_plausible_match_count',
    'evaluation_unit_id', 'catalog_detection_extension_status',
    'repeater_family_extension_status'
]
display(view[columns])

## Decision and next gate

This checkpoint sharpens the comparator but does **not** establish a catalog extension:

- six network rows are catalog vetoes, representing five known earthquakes;
- 27 rows remain candidates, not confirmed earthquakes;
- repeating-family membership was not evaluated;
- the auxiliary generic branch retains its development SNR=1 STOP;
- scientific extension remains STOP until the DAS-v2 detector runs independently over all 12 complete intervals.

The next permitted step is a separately registered DAS replay whose candidate generation cannot read these network or catalog times. Only after both candidate tables are frozen may we measure DAS-only additions, network-only events, matched events, and any later improvement in family resolution.